# RK T4 Rev 5.0 EVT — 07 — CPU-Only Acoustics (Elmer Helmholtz + Klippel TS ODE), LRA Haptics vs OIS & Touch Electrostatics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sc8z35a-collab/GIT-Y5/blob/main/colab_ultimate_signoff/07_cpu_acoustics_haptics_touch.ipynb)

| Field | Value |
|---|---|
| **Recommended Colab Pro Runtime** | `High-RAM CPU` |
| **Execution Wave (DAG)** | `Wave 2B (After NB 01; run alongside 03, 04a, 08)` |
| **Estimated Duration** | `QUICK_CHECK: ~8 min | ULTRA_SIGNOFF: ~30–45 min` |
| **Source Repository** | `https://github.com/sc8z35a-collab/GIT-Y5` (`main`) |
| **Read-Only Guarantee** | All solver work runs in `/tmp/rk_t4_work/` (auto-GC'd per case) & persists atomically to Google Drive; tracked design files remain 100% unmodified. |

### Verification Scope
1. **Role-Separated Acoustic Simulation (Zero Physical Contradictions)**:
   - **`Elmer FEM` `HelmholtzSolver`** with waterproof ePTFE lumped acoustic impedance boundary: 200 Hz–18 kHz SPL curve, L/R polarity, MEMS mic boot seal integrity, and Speaker→Mic ERLE (`>= 18 dB`).
   - **Nonlinear Klippel Thiele-Small ODE ($Bl(x), K_{ms}(x)$)**: verifies peak diaphragm excursion $X_{\max}$ stays strictly clear of front mesh and THD `<= 10%`.
2. **LRA Haptics (175 Hz) vs Camera OIS (50–120 Hz) Resonance Decoupling (`CalculiX`)**.
3. **Capacitive Touch SNR, Water-Droplet Bridge & Charger Common-Mode Noise Immunity (`Elmer FEM` `StatElecSolver`)**.


In [ ]:
#@title 1. Standalone Repository Self-Clone, Google Drive Mount & Hardware Bootstrap
REPO_URL = "https://github.com/sc8z35a-collab/GIT-Y5.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
RUN_MODE = "ULTRA_SIGNOFF" #@param ["QUICK_CHECK", "ULTRA_SIGNOFF"]
BATCH_INDEX = 0 #@param {type:"integer"}
RANDOM_SEED = 42 #@param {type:"integer"}
DRIVE_ROOT = "/content/drive/MyDrive/RK_T4_VirtualLab" #@param {type:"string"}

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/rk_t4_repo")
if not (REPO_DIR / ".git").exists():
    clone_url = REPO_URL
    token = os.environ.get("GITHUB_TOKEN", "")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN") or ""
        except Exception:
            token = ""
    if token and clone_url.startswith("https://github.com/"):
        clone_url = clone_url.replace("https://github.com/", f"https://{token}@github.com/", 1)
    res = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(REPO_DIR)],
        text=True,
        capture_output=True,
    )
    if res.returncode != 0:
        safe_err = res.stderr.replace(token, "***") if token else res.stderr
        raise RuntimeError(f"Standalone git clone failed: {safe_err}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from colab_ultimate_signoff.common.colab_bootstrap import (
    bootstrap_notebook,
    ensure_apt_tools,
    ensure_cpu_cae_env,
    run_in_env,
    verify_repo_unmodified,
    verify_tool_smoke,
)
from colab_ultimate_signoff.common.schemas import validate_report_file

ctx = bootstrap_notebook(
    repo_url=REPO_URL,
    repo_branch=REPO_BRANCH,
    repo_dir=REPO_DIR,
    drive_root=DRIVE_ROOT,
    require_gpu=False,
)
print(f"[NB 07] Repo Commit : {ctx.git_commit_sha}")
print(f"[NB 07] Runtime HW  : OS={ctx.hw.os_codename}, CPU={ctx.hw.cpu_count} (OMP={ctx.hw.omp_threads}), RAM={ctx.hw.ram_gb} GB, GPU={ctx.hw.gpu_name or 'None'}")
print(f"[NB 07] Mode / Seed : RUN_MODE={RUN_MODE}, BATCH_INDEX={BATCH_INDEX}, SEED={RANDOM_SEED}")
print(f"[NB 07] Drive Root  : {ctx.drive_root} (tmp free={ctx.hw.disk_free_tmp_gb} GB)")


In [ ]:
#@title 2. Restore / Install Elmer FEM + CalculiX (`ccx`) + CPU CAE Env & Smoke-Test
ensure_apt_tools(ctx, need_elmer=True, need_calculix=True)
ensure_cpu_cae_env(ctx, need_elmer_conda=True)
tool_versions = verify_tool_smoke(["elmer", "ccx", "cpu_cae_python"], profile="cpu_cae")
for k, v in tool_versions.items():
    print(f"  [OK] {k:22s}: {v}")


In [ ]:
#@title 3. Execute Elmer Acoustics/Touch + Klippel Nonlinear ODE + CalculiX Haptics Suite
worker = ctx.repo_dir / "colab_ultimate_signoff" / "workers" / "nb07_acoustics_haptics_touch_worker.py"
proc = run_in_env(
    profile="cpu_cae",
    script_path=worker,
    args=[
        "--repo-dir", str(ctx.repo_dir),
        "--drive-root", str(ctx.drive_root),
        "--run-mode", RUN_MODE,
        "--git-sha", ctx.git_commit_sha,
        "--seed", str(RANDOM_SEED),
    ],
    cwd=ctx.repo_dir,
    timeout_s=3600,
)
print(proc.stdout)


In [ ]:
#@title 4. Strict Schema Validation, Visual Inspection & Read-Only Repo Integrity Proof
from pathlib import Path
from IPython.display import Image, display

report_path = ctx.reports_dir / "nb07_report.json"
report = validate_report_file(report_path)

print("=" * 84)
print(f"RK T4 DIGITAL TWIN REPORT — NB {report.notebook_id} ({report.notebook_name})")
print(f"Overall Status : {report.overall_status} | Run Mode : {report.run_mode} | Fidelity : {report.model_fidelity}")
print(f"Git Commit SHA : {report.git_commit_sha} | Seed : {report.random_seed}")
print("=" * 84)
for chk in report.checks:
    print(f"[{chk.status:4s}] {chk.check_id:44s} | meas={str(chk.measured_value)[:36]:36s} | lim={chk.limit_value} ({chk.unit})")

print("\n--- Solver Provenance Receipts ---")
for prov in report.solver_provenance:
    bb = prov.model_bbox_mm
    print(
        f"  * {prov.solver_name} ({prov.solver_version}) | scope={prov.model_scope} "
        f"| nodes={prov.mesh_nodes}, elems={prov.mesh_elements} "
        f"| bbox={bb.span_x_mm:.1f}x{bb.span_y_mm:.1f}x{bb.span_z_mm:.1f}mm "
        f"| wall={prov.solve_wall_time_s:.2f}s | files={len(prov.raw_output_files)}"
    )

for key, art_path in report.handoff_artifacts.items():
    p = Path(art_path)
    if p.suffix.lower() == ".png" and p.is_file():
        print(f"--- Figure: {key} ({p.name}) ---")
        display(Image(filename=str(p)))

# MANDATORY Read-Only Invariance Check (tracked design files must be 100% untouched!)
assert verify_repo_unmodified(ctx.repo_dir), "Repository tracked design files were modified!"
print("\n[READ-ONLY VERIFIED] git diff --quiet HEAD -- hardware/ firmware/ src/ optics/ mechanics/ PASSED (0 changes).")
